In [ ]:
from process_eeg import get_psd_df, get_psd_avg_df, get_band_metrics_df
from viz.vis_eeg import compare_epo_psd, compare_band_metric, compare_found_peaks
from spanav_eeg_utils.io_utils import get_sids
import numpy as np

%load_ext autoreload
%autoreload 2

In [ ]:
test = False
show = True
save = False

log = True
kwargs = {
    'show': show, 
    'save': save,
}

# Overview of all epoch-types
## PSD

In [ ]:
# Load dataframe with power spectra
psd_df = get_psd_df(test=test, log=log)

In [ ]:
# Load dataframe with average power spectra
psd_avg_df = get_psd_avg_df(log=log)

In [ ]:
# Average across subjects
compare_epo_psd(psd_avg_df, 'epo_type', plot_subj='average', **kwargs)

In [ ]:
# For each subject separately  
psd_df_lin = get_psd_df(test=test, log=False)  # need linear psd dfs to compute std across blocks
for sid in get_sids():
    df = psd_df_lin[psd_df_lin['sid'] == sid]
    df['pw_avg'] = np.log10(df['pw_avg'])
    plot_df = df.groupby(['cond', 'epo_type', 'freq'], as_index=False).agg(  # average together blocks of the same condition
            sid=("sid", 'first'),
            freq=("freq", 'first'),
            pw_avg=('pw_avg', 'mean'),
            pw_std=('pw_avg', 'std'),
            N=('pw_avg', 'count'),
        )
    compare_epo_psd(plot_df, 'epo_type', plot_subj=sid, **kwargs)  # sem_n is number of blocks by condition

## Oscillations

In [ ]:
# Load dataframe with power spectra
osc_df = get_band_metrics_df(test=test)

In [ ]:
style_kwargs = dict(
    x_by='band', 
    facet_by='cond', 
    color_by='epo_type',
)

## Power in canonical bands

In [ ]:
compare_band_metric(osc_df, metric_name='abs_pw', **style_kwargs, **kwargs)
compare_band_metric(osc_df, metric_name='rel_pw', **style_kwargs, **kwargs)

In [ ]:
# For each subject separately
for sid in get_sids(test):
    df = osc_df[osc_df['sid'] == sid]
    compare_band_metric(df, metric_name='abs_pw', **style_kwargs, **kwargs)
    compare_band_metric(df, metric_name='rel_pw', **style_kwargs, **kwargs)

## SNR 
Based on oscillations / background noise

In [ ]:
# Average across patients
compare_band_metric(osc_df, metric_name='osc_snr', **style_kwargs, **kwargs)

In [ ]:
# For each subject separately
for sid in get_sids(test):
    df = osc_df[osc_df['sid'] == sid]
    compare_band_metric(df, metric_name='osc_snr', **style_kwargs, **kwargs)

# Peaks

In [ ]:
compare_found_peaks(osc_df, metric_name='pk', **kwargs)